## Gerar_clientes_propensos_ao_churn
--------------------------------------
Treina um modelo de Machine Learning (Random Forest) para prever churn e
gera uma tabela com os clientes ATIVOS (que ainda não cancelaram) mais
propensos a cancelar o serviço em seguida.

O QUE O SCRIPT FAZ:
  1. Carrega e limpa a base de dados (data/Telecom_Churn.csv)
  2. Treina um Random Forest com os hiperparâmetros otimizados
     (max_depth=9, n_estimators=100)
  3. Aplica o modelo em todos os clientes ativos (Churn == 'No')
  4. Calcula a probabilidade de churn de cada um
  5. Classifica em faixas de risco (Alto / Médio / Baixo)
  6. Identifica os principais fatores de risco de cada cliente
  7. Exporta o resultado em CSV e em Excel formatado (com cores e filtros)

COMO RODAR:
  1. Instale as dependências (uma vez só):
       pip install pandas numpy scikit-learn openpyxl

  2. Coloque o arquivo "Telecom_Churn.csv" na mesma pasta deste script,
     OU ajuste a variável CAMINHO_DADOS abaixo com o caminho correto.

  3. Rode:
       python gerar_clientes_propensos_ao_churn.py

SAÍDA:
  - clientes_propensos_ao_churn.csv
  - clientes_propensos_ao_churn.xlsx  (aba "Resumo" + aba "Clientes em Risco")
"""


In [35]:
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
 
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import CellIsRule, DataBarRule
from openpyxl.worksheet.table import Table, TableStyleInfo
 
pd.set_option('display.max_columns', None)

In [36]:

# ============================================================
# CONFIGURAÇÕES — ajuste aqui se necessário
# ============================================================
# Caminho da base de dados: a pasta "data/" fica um nível acima de "notebooks/"
CAMINHO_DADOS = 'Telecom_Churn.csv'
 
SAIDA_CSV = 'outputs/clientes_propensos_ao_churn.csv'
SAIDA_XLSX = 'outputs/clientes_propensos_ao_churn.xlsx'
 
LIMIAR_ALTO_RISCO = 0.50   # probabilidade >= 50% -> Alto risco
LIMIAR_MEDIO_RISCO = 0.25  # probabilidade >= 25% -> Médio risco (senão, Baixo)
 

In [37]:
# ============================================================
# 1) CARREGAR E TRATAR OS DADOS
# ============================================================
def carregar_dados(caminho):
    if not os.path.isfile(caminho):
        raise FileNotFoundError(
            f"\nArquivo não encontrado: '{caminho}'\n"
            f"Pasta atual: {os.getcwd()}\n"
            f"Dica: confirme se está rodando a partir da pasta 'notebooks/', "
            f"ou ajuste CAMINHO_DADOS no início do script."
        )
    df = pd.read_csv(caminho)
 
    # Corrigir TotalCharges (vem como texto por causa de valores em branco)
    df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
 
    # Garantir consistência do rótulo alvo
    df['Churn'] = df['Churn'].replace({0: 'No', 1: 'Yes'})
 
    return df
 

In [38]:
 
# ============================================================
# 2) TREINAR O MODELO
# ============================================================
def treinar_modelo(df):
    X = df.drop(columns=['customerID', 'Churn'])
    y_raw = df['Churn']
 
    le = LabelEncoder()
    y = le.fit_transform(y_raw)  # No -> 0, Yes -> 1
 
    X = pd.get_dummies(X)
    feature_names = list(X.columns)
 
    mm = MinMaxScaler()
    X_scaled = pd.DataFrame(mm.fit_transform(X), columns=feature_names)
 
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.25, random_state=42, stratify=y
    )
 
    # Random Forest com os melhores hiperparâmetros (encontrados via GridSearchCV)
    modelo = RandomForestClassifier(max_depth=9, n_estimators=100, random_state=42)
    modelo.fit(X_train, y_train)
 
    acuracia_teste = modelo.score(X_test, y_test)
    print(f"Acurácia do modelo no conjunto de teste: {acuracia_teste:.2%}")
 
    return modelo, X_scaled

In [39]:

 
# ============================================================
# 3) PONTUAR CLIENTES ATIVOS E MONTAR A TABELA FINAL
# ============================================================
def faixa_risco(p):
    if p >= LIMIAR_ALTO_RISCO:
        return 'Alto'
    elif p >= LIMIAR_MEDIO_RISCO:
        return 'Médio'
    return 'Baixo'
 
 
def fatores_risco(row):
    fatores = []
    if row['Contract'] == 'Month-to-month':
        fatores.append('Contrato mensal')
    if row['tenure'] <= 12:
        fatores.append('Cliente novo (≤12 meses)')
    if row['InternetService'] == 'Fiber optic':
        fatores.append('Internet fibra óptica')
    if row['PaymentMethod'] == 'Electronic check':
        fatores.append('Pagamento por cheque eletrônico')
    if row['MonthlyCharges'] > 70:
        fatores.append('Mensalidade alta')
    return ', '.join(fatores) if fatores else '—'
 
 
def montar_tabela_risco(df, modelo, X_scaled):
    # Probabilidade de churn para TODOS os clientes
    df['Probabilidade_Churn'] = modelo.predict_proba(X_scaled)[:, 1]
 
    # Filtrar apenas clientes ATIVOS (ainda não cancelaram)
    ativos = df[df['Churn'] == 'No'].copy()
    ativos['Faixa_Risco'] = ativos['Probabilidade_Churn'].apply(faixa_risco)
    ativos['Principais_Fatores_de_Risco'] = ativos.apply(fatores_risco, axis=1)
 
    colunas_finais = [
        'customerID', 'Probabilidade_Churn', 'Faixa_Risco',
        'Principais_Fatores_de_Risco', 'tenure', 'Contract',
        'InternetService', 'MonthlyCharges', 'TotalCharges',
        'PaymentMethod', 'PaperlessBilling', 'gender',
        'SeniorCitizen', 'Partner', 'Dependents',
    ]
    resultado = ativos[colunas_finais].rename(columns={
        'customerID': 'ID_Cliente',
        'tenure': 'Meses_de_Contrato',
        'Contract': 'Tipo_Contrato',
        'InternetService': 'Servico_Internet',
        'MonthlyCharges': 'Mensalidade',
        'TotalCharges': 'Total_Pago',
        'PaymentMethod': 'Forma_Pagamento',
        'PaperlessBilling': 'Fatura_Sem_Papel',
        'gender': 'Genero',
        'SeniorCitizen': 'Idoso',
        'Partner': 'Tem_Parceiro',
        'Dependents': 'Tem_Dependentes',
    })
 
    resultado = resultado.sort_values('Probabilidade_Churn', ascending=False).reset_index(drop=True)
    resultado.insert(0, 'Ranking', range(1, len(resultado) + 1))
    resultado['Probabilidade_Churn'] = (resultado['Probabilidade_Churn'] * 100).round(1)
 
    return resultado
 

In [40]:
 
# ============================================================
# 4) EXPORTAR CSV
# ============================================================
def exportar_csv(resultado, caminho):
    resultado.to_csv(caminho, index=False, encoding='utf-8-sig')
    print(f"CSV salvo em: {caminho}")
 
 

In [41]:
# ============================================================
# 5) EXPORTAR EXCEL FORMATADO (aba Resumo + aba Clientes em Risco)
# ============================================================
def exportar_excel(resultado, caminho):
    resumo_risco = (
        resultado['Faixa_Risco'].value_counts()
        .reindex(['Alto', 'Médio', 'Baixo'])
        .reset_index()
    )
    resumo_risco.columns = ['Faixa_Risco', 'Qtd_Clientes']
 
    wb = Workbook()
    FONT_NAME = 'Arial'
    risco_cores = {'Alto': 'E74C3C', 'Médio': 'F39C12', 'Baixo': '27AE60'}
 
    # ---------- Aba 1: Resumo ----------
    ws1 = wb.active
    ws1.title = 'Resumo'
 
    title_font = Font(name=FONT_NAME, size=16, bold=True, color='1A1A2E')
    subtitle_font = Font(name=FONT_NAME, size=10, italic=True, color='666666')
    header_font = Font(name=FONT_NAME, size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill('solid', fgColor='1A1A2E')
    normal_font = Font(name=FONT_NAME, size=11)
    bold_font = Font(name=FONT_NAME, size=11, bold=True)
 
    ws1['B2'] = 'Clientes Propensos ao Churn — Telecom'
    ws1['B2'].font = title_font
    ws1['B3'] = 'Score de risco de cancelamento gerado por modelo de Machine Learning (Random Forest)'
    ws1['B3'].font = subtitle_font
    ws1.merge_cells('B2:F2')
    ws1.merge_cells('B3:F3')
 
    ws1['B5'] = 'Total de clientes ativos avaliados'
    ws1['B5'].font = bold_font
    ws1['C5'] = int(resumo_risco['Qtd_Clientes'].sum())
    ws1['C5'].font = normal_font
 
    row = 7
    for col, texto in zip('BCD', ['Faixa de Risco', 'Qtd. Clientes', '% da Base Ativa']):
        c = ws1[f'{col}{row}']
        c.value = texto
        c.font = header_font
        c.fill = header_fill
 
    total_ativos = resumo_risco['Qtd_Clientes'].sum()
    for _, r in resumo_risco.iterrows():
        row += 1
        c_risco = ws1.cell(row=row, column=2, value=r['Faixa_Risco'])
        c_qtd = ws1.cell(row=row, column=3, value=int(r['Qtd_Clientes']))
        c_pct = ws1.cell(row=row, column=4, value=round(r['Qtd_Clientes'] / total_ativos * 100, 1))
        c_pct.number_format = '0.0"%"'
        c_risco.fill = PatternFill('solid', fgColor=risco_cores.get(r['Faixa_Risco'], 'FFFFFF'))
        c_risco.font = Font(name=FONT_NAME, color='FFFFFF', bold=True)
        c_qtd.font = normal_font
        c_pct.font = normal_font
        for c in (c_risco, c_qtd, c_pct):
            c.alignment = Alignment(horizontal='center')
 
    row += 3
    ws1.cell(row=row, column=2, value='Critério de classificação de risco:').font = bold_font
    for texto in [
        'Alto risco: probabilidade de churn ≥ 50%',
        'Médio risco: probabilidade de churn entre 25% e 50%',
        'Baixo risco: probabilidade de churn < 25%',
    ]:
        row += 1
        ws1.cell(row=row, column=2, value=texto).font = normal_font
 
    row += 3
    ws1.cell(row=row, column=2, value='Modelo utilizado:').font = bold_font
    row += 1
    ws1.cell(row=row, column=2,
             value='Random Forest (max_depth=9, n_estimators=100)').font = normal_font
    row += 1
    ws1.cell(row=row, column=2,
             value=f'Base: {total_ativos} clientes atualmente ativos (que ainda não cancelaram)').font = normal_font
 
    row += 3
    ws1.cell(row=row, column=2,
             value='➜ Ver a lista completa e ordenada de clientes na aba "Clientes em Risco".').font = \
        Font(name=FONT_NAME, size=11, italic=True, color='1A1A2E')
 
    ws1.column_dimensions['A'].width = 3
    ws1.column_dimensions['B'].width = 42
    ws1.column_dimensions['C'].width = 16
    ws1.column_dimensions['D'].width = 16
 
    # ---------- Aba 2: Clientes em Risco ----------
    ws2 = wb.create_sheet('Clientes em Risco')
 
    col_headers = list(resultado.columns)
    header_map = {
        'Ranking': 'Ranking',
        'ID_Cliente': 'ID do Cliente',
        'Probabilidade_Churn': 'Probabilidade de Churn (%)',
        'Faixa_Risco': 'Faixa de Risco',
        'Principais_Fatores_de_Risco': 'Principais Fatores de Risco',
        'Meses_de_Contrato': 'Meses de Contrato',
        'Tipo_Contrato': 'Tipo de Contrato',
        'Servico_Internet': 'Serviço de Internet',
        'Mensalidade': 'Mensalidade (R$)',
        'Total_Pago': 'Total Pago (R$)',
        'Forma_Pagamento': 'Forma de Pagamento',
        'Fatura_Sem_Papel': 'Fatura Sem Papel',
        'Genero': 'Gênero',
        'Idoso': 'Idoso (1=Sim)',
        'Tem_Parceiro': 'Tem Parceiro(a)',
        'Tem_Dependentes': 'Tem Dependentes',
    }
    display_headers = [header_map.get(c, c) for c in col_headers]
 
    for j, h in enumerate(display_headers, start=1):
        cell = ws2.cell(row=1, column=j, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
 
    thin = Side(style='thin', color='DDDDDD')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
 
    for i, (_, r) in enumerate(resultado.iterrows(), start=2):
        for j, col in enumerate(col_headers, start=1):
            cell = ws2.cell(row=i, column=j, value=r[col])
            cell.font = Font(name=FONT_NAME, size=10)
            cell.border = border
            if col == 'Probabilidade_Churn':
                cell.number_format = '0.0"%"'
                cell.alignment = Alignment(horizontal='center')
            elif col in ('Mensalidade', 'Total_Pago'):
                cell.number_format = '#,##0.00'
                cell.alignment = Alignment(horizontal='right')
            elif col in ('Ranking', 'Meses_de_Contrato', 'Idoso', 'Faixa_Risco'):
                cell.alignment = Alignment(horizontal='center')
 
    faixa_col_idx = col_headers.index('Faixa_Risco') + 1
    faixa_col_letter = get_column_letter(faixa_col_idx)
    last_row = len(resultado) + 1
 
    for valor, cor in risco_cores.items():
        rule = CellIsRule(
            operator='equal', formula=[f'"{valor}"'],
            fill=PatternFill('solid', fgColor=cor),
            font=Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
        )
        ws2.conditional_formatting.add(f'{faixa_col_letter}2:{faixa_col_letter}{last_row}', rule)
 
    prob_col_idx = col_headers.index('Probabilidade_Churn') + 1
    prob_col_letter = get_column_letter(prob_col_idx)
    databar_rule = DataBarRule(
        start_type='num', start_value=0, end_type='num', end_value=100,
        color='E74C3C', showValue=True,
    )
    ws2.conditional_formatting.add(f'{prob_col_letter}2:{prob_col_letter}{last_row}', databar_rule)
 
    ws2.freeze_panes = 'C2'
 
    larguras = {
        'Ranking': 9, 'ID_Cliente': 14, 'Probabilidade_Churn': 14,
        'Faixa_Risco': 13, 'Principais_Fatores_de_Risco': 50,
        'Meses_de_Contrato': 12, 'Tipo_Contrato': 16, 'Servico_Internet': 15,
        'Mensalidade': 14, 'Total_Pago': 14, 'Forma_Pagamento': 22,
        'Fatura_Sem_Papel': 14, 'Genero': 10, 'Idoso': 12,
        'Tem_Parceiro': 13, 'Tem_Dependentes': 15,
    }
    for j, col in enumerate(col_headers, start=1):
        ws2.column_dimensions[get_column_letter(j)].width = larguras.get(col, 14)
 
    n_rows = len(resultado) + 1
    last_col_letter = get_column_letter(len(col_headers))
    tab = Table(displayName='ClientesEmRisco', ref=f'A1:{last_col_letter}{n_rows}')
    tab.tableStyleInfo = TableStyleInfo(
        name='TableStyleMedium2', showFirstColumn=False,
        showLastColumn=False, showRowStripes=True, showColumnStripes=False,
    )
    ws2.add_table(tab)
 
    wb.save(caminho)
    print(f"Excel salvo em: {caminho}")
 
 
# ============================================================
# MAIN
# ============================================================
def main():
    # garante que a pasta de saída existe (ex: ../outputs) antes de salvar os arquivos
    os.makedirs(os.path.dirname(SAIDA_CSV), exist_ok=True)
 
    print("1/5 — Carregando os dados...")
    df = carregar_dados(CAMINHO_DADOS)
 
    print("2/5 — Treinando o modelo (Random Forest)...")
    modelo, X_scaled = treinar_modelo(df)
 
    print("3/5 — Pontuando clientes ativos e montando a tabela de risco...")
    resultado = montar_tabela_risco(df, modelo, X_scaled)
 
    print(f"\nTotal de clientes ativos avaliados: {len(resultado)}")
    print(resultado['Faixa_Risco'].value_counts())
    print("\nTop 10 clientes de maior risco:")
    print(resultado.head(10)[['Ranking', 'ID_Cliente', 'Probabilidade_Churn', 'Faixa_Risco']])
 
    print("\n4/5 — Exportando CSV...")
    exportar_csv(resultado, SAIDA_CSV)
 
    print("5/5 — Exportando Excel formatado...")
    exportar_excel(resultado, SAIDA_XLSX)
 
    print("\nConcluído! Arquivos gerados:")
    print(f"  - {SAIDA_CSV}")
    print(f"  - {SAIDA_XLSX}")
 
 
if __name__ == '__main__':
    main()
 

1/5 — Carregando os dados...


FileNotFoundError: 
Arquivo não encontrado: 'Telecom_Churn.csv'
Pasta atual: c:\Users\vivid\Previs-o-de-churn-de-clientes--Telecom\notebooks
Dica: confirme se está rodando a partir da pasta 'notebooks/', ou ajuste CAMINHO_DADOS no início do script.

1/2 — Carregando os 10 clientes de maior risco...
 Ranking ID_Cliente  Probabilidade_Churn Faixa_Risco
       1 4912-PIGUY                 86.8        Alto
       2 1452-VOQCH                 83.8        Alto
       3 2545-EBUPK                 82.9        Alto
       4 7439-DKZTW                 82.4        Alto
       5 2254-DLXRI                 81.4        Alto
       6 5150-ITWWB                 81.1        Alto
       7 4927-WWOOZ                 81.0        Alto
       8 8161-QYMTT                 80.9        Alto
       9 8622-ZLFKO                 80.3        Alto
      10 1628-BIZYP                 80.0        Alto

2/2 — Gerando gráfico...

Concluído! Gráfico salvo em: c:\Users\vivid\Previs-o-de-churn-de-clientes--Telecom\notebooks\outputs\top10_clientes_risco.png
